In [7]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# ---- data: Statistics Jersey National Accounts, "GVA (Market) by sub-sector - combined dataset"
raw = pd.read_csv("../Data/new-sic2007-components-of-gva-2025.csv", encoding="utf-8-sig")
raw.columns = raw.columns.str.strip()
for c in ["Type", "Sector Code", "Division Description", "Measure"]:
    raw[c] = raw[c].astype(str).str.strip()

p = (raw[raw["Type"] == "Real-Term"]
     .pivot_table(index=["Sector Code", "Division Description", "Year"],
                  columns="Measure", values="Value", aggfunc="sum")
     .reset_index())
allec = p[p["Sector Code"] == "All"].set_index("Year")
sub = p[p["Sector Code"] != "All"]

w = sub[sub["FTE"] > 0].copy()                          # official basis: drop sectors with no FTE
w["GVA"] = w["GVA"] / 1e6                               # £m
w["gva_per_fte"] = w["GVA"] * 1000 / w["FTE"]           # £000 per FTE, 2025 prices
w["date"] = pd.to_datetime(w["Year"], format="%Y")
zero_gva = sub[~(sub["FTE"] > 0)].groupby("Year")["GVA"].sum() / 1e6

# ---- checks: rows must rebuild the All Economy totals, and the official productivity figure
tot = w.groupby("Year")[["GVA", "FTE"]].sum()
assert ((tot.GVA + zero_gva) - allec["GVA"] / 1e6).abs().max() < 0.01, "GVA does not tie out"
assert (tot.FTE - allec["FTE"]).abs().max() < 0.01, "FTE does not tie out"
prod = (tot.GVA * 1e6 / tot.FTE).round(0)
print("GVA per FTE, official basis (£):", prod.to_dict())          # published: 2024 about 107,100; 2025 about 104,500
print("2024 to 2025: %+.1f%% (published -2.5%%)" % ((prod[2025] / prod[2024] - 1) * 100))

GVA per FTE, official basis (£): {2021: 94804.0, 2022: 99602.0, 2023: 108615.0, 2024: 107144.0, 2025: 104459.0}
2024 to 2025: -2.5% (published -2.5%)


In [10]:
import numpy as np

# ---- annual decomposition, as % of start-year GVA per FTE  (uses w and shiftshare from above)
periods = [(2021, 2022), (2022, 2023), (2023, 2024), (2024, 2025)]
components = ["within", "between", "interaction"]

def decomp_table(df, panel):
    rows = []
    for y0, y1 in periods:
        s = shiftshare(df, y0, y1)
        for comp in components:
            rows.append({"panel": panel, "period": f"{y0}-{str(y1)[2:]}", "component": comp,
                         "pct": s[comp] / s["P0"] * 100})
    return pd.DataFrame(rows)

no_bank = w[w["Division Description"] != "Monetary intermediation"]
d = pd.concat([decomp_table(w, "All sub-sectors"), decomp_table(no_bank, "Excluding banking")])
total = d.groupby(["panel", "period"], as_index=False)["pct"].sum()
print(d.pivot_table(index=["panel", "period"], columns="component", values="pct")
       .assign(total=total.set_index(["panel", "period"])["pct"]).round(1))

pos = d[d.pct > 0].groupby(["panel", "period"])["pct"].sum().max()
neg = d[d.pct < 0].groupby(["panel", "period"])["pct"].sum().min()
ylim = [float(np.floor(min(neg, total.pct.min()) - 1)), float(np.ceil(max(pos, total.pct.max()) + 1))]

def bars(panel, first):
    dd, tt = d[d.panel == panel], total[total.panel == panel]
    b = alt.Chart(dd).mark_bar().encode(
        x=alt.X("period:N", axis=alt.Axis(labelAngle=0), title=None),
        y=alt.Y("pct:Q", scale=alt.Scale(domain=ylim),
                title="Contribution to change in GVA per FTE (percentage points)" if first else None),
        color=alt.Color("component:N", sort=components,
                        legend=alt.Legend(orient="bottom", title=None, labelFontSize=11)))
    t = alt.Chart(tt).mark_tick(color="black", thickness=2.5, size=42).encode(x="period:N", y="pct:Q")
    return alt.layer(b, t).properties(width=300, height=380,
                                      title=alt.Title(panel, fontSize=12, anchor="start"))

decomp_chart = (alt.hconcat(bars("All sub-sectors", True), bars("Excluding banking", False), spacing=40)

    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

decomp_chart.save("jersey_productivity_decomposition.svg")
decomp_chart.save("jersey_productivity_decomposition.png", ppi=288)
decomp_chart

component                  between  interaction  within  total
panel             period                                      
All sub-sectors   2021-22     -1.2         -0.5     6.7    5.1
                  2022-23     -1.4         -0.7    11.1    9.0
                  2023-24     -0.3         -0.0    -1.0   -1.4
                  2024-25     -0.6          0.1    -1.9   -2.5
Excluding banking 2021-22     -0.6         -0.0    -2.4   -3.1
                  2022-23     -0.2         -0.1     2.7    2.4
                  2023-24     -0.2         -0.1     3.9    3.7
                  2024-25      0.3         -0.0    -0.7   -0.4


alt.HConcatChart(...)

In [9]:
def contrib(w, y0, y1):
    a = w[w.Year == y0].set_index("Division Description"); b = w[w.Year == y1].set_index("Division Description")
    idx = a.index.intersection(b.index); a, b = a.loc[idx], b.loc[idx]
    s0, s1 = a.FTE / a.FTE.sum(), b.FTE / b.FTE.sum()
    p0, p1 = a.GVA * 1e6 / a.FTE, b.GVA * 1e6 / b.FTE
    P0 = a.GVA.sum() * 1e6 / a.FTE.sum()
    return pd.DataFrame({"fte_share_change_pp": (s1 - s0) * 100, "prod_start_k": p0 / 1000,
                         "within": s0 * (p1 - p0), "between": (s1 - s0) * (p0 - P0)})

print(contrib(w, 2021, 2025).sort_values("between").round(1))

                                                    fte_share_change_pp  \
Division Description                                                      
Monetary intermediation                                            -0.8   
Food and beverage service activities                                0.7   
Trust administration, trustee and fiduciary ser...                 -0.6   
Public administration                                               1.5   
Arts, entertainment and recreation                                  0.3   
Transportation and Storage                                          0.2   
Construction of buildings                                          -0.5   
Other Service Activities                                            0.1   
Private sector human health and social work act...                  0.1   
Accommodation                                                       0.0   
Administrative and support service activities                       0.0   
Fishing and aquaculture  